### 네이버 뉴스 안정성을 높인 수집기

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import unquote

from datetime import datetime

In [ ]:
ds = de = datetime.now().strftime('%Y.%m.%d')

'2026.08.25'

In [ ]:
URL = 'https://s.search.naver.com/p/newssearch/3/api/tab/more'

param_str = 'abt=null&de=2026.08.24&ds=2026.08.24&eid=&field=0&force_original=&is_dts=0&is_sug_officeid=0&mynews=0&news_office_checked=&nlu_query=&nqx_theme=%7B%22theme%22%3A%7B%22main%22%3A%7B%22name%22%3A%22encyclopedia%22%2C%22source%22%3A%22TOS%22%7D%7D%7D&nso=so%3Add%2Cp%3Aall%2Ca%3Aall&nx_and_query=&nx_search_hlquery=&nx_search_query=&nx_sub_query=&office_category=0&office_section_code=0&office_type=0&pd=0&photo=0&qdt=0&query=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5&query_original=&rev=0&service_area=0&sm=tab_smr&sort=1&spq=0&ssc=tab.news.all&start=11'
params = [p.split('=') for p in param_str.split('&')]
params = {k: unquote(v) for k, v in params}

keyword = '인공지능'
start = 1 # 1, 11, 21, ...
ds = de = None

from requests.exceptions import Timeout, ConnectionError

class StatusCodeException(Exception):
    ...

class RetryException(Exception):
    ...

import time

# fetch - 재시도 + 예외처리
def fetch_list(keyword, start, ds=None, de=None, max_retires=4):
    if ds == de == None:
        ds = de = datetime.now().strftime('%Y.%m.%d')

    for i in range(max_retires):
        try:
            res = requests.get(URL, 
                            params={**params, 'query': keyword, 'start': start,
                                    'ds': ds, 'de': de})

            # 재시도 안함.
            if res.status_code in (400, 401, 403, 404):
                raise StatusCodeException('Fetch 재시도 안하는 오류')
            elif res.status_code//100 == 5 or res.status_code == 429:
                # 재시도
                raise RetryException('재시도해야하는 오류')

            return res
        except (Timeout, ConnectionError) as e:
            logger.warning(f'요청(timeout) 에러: {e}')
            time.sleep(2**i) # 지수 백오프로 1 -> 2 -> 4 -> 8
        except RetryException as e:
            logger.info('재시도 오류')
            time.sleep(2**i)
        # except Exception as e:
        #     print(f'[WARN] 알수없는 에러: {e}')

        logger.info(f'재시도 횟수 {i+1}번 째')
    else:
        raise RetryException('Fetch 재시도 횟수 초과')

def fetch_detail(url, ):
    
    return

In [ ]:
class ParsingError(Exception):
    ...
    
def parse_list(res) -> list[dict]:
    try:
        soup = BeautifulSoup(res.json()['collection'][0]['html'], 'html.parser')
        items = soup.select('.fds-news-item-list-tab > div')

        news = []
        for item in items:
            # 링크
            # 네이버뉴스 체크
            naver_news_link = item.select_one('a[href^="https://n.news.naver.com"]')
            if naver_news_link:
                link = naver_news_link.attrs['href']
            else:
                # link 다시 찾아오기
                link = item.select_one('a[data-heatmap-target=".tit"]').attrs['href']

            news.append({
                '제목': item.select_one('.sds-comps-text-type-headline1').text.strip(),
                # '언론사': item.select_one('.sds-comps-profile-info-title').text.replace('새 창 열림', '').strip(),
                '언론사': item.select_one('.sds-comps-profile-info-title a span').text.strip(),
                '링크': link,
                '요약': item.select_one('.sds-comps-text-type-body1').text.strip(),
            })

        return news
    except Exception as e:
        raise ParsingError(f'Parsing 에러, {e}')



In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
    logging.FileHandler("crawler.log", encoding="utf-8"),
    logging.StreamHandler(),
    ],
)

logger = logging.getLogger('crawler')

In [ ]:
%pip install tqdm

In [ ]:
from tqdm import tqdm

stats = {'ok': 0, 'failed': 0, 'skip': 0}
for page in tqdm(range(10), desc="네이버 뉴스 목록 수집"):
    try:
        res = fetch_list('인공지능', page*10+1)
    except Exception as e:
        logger.warning(f'fetch 에러 발생: {e}')
        stats['skip'] += 1
        continue
    finally:
        time.sleep(0.5) # 매너

    try:
        news = parse_list(res)
        stats['ok'] += 1
    except Exception as e:
        stats['skip'] += 1
        logger.warning(f'parsing 에러, {e}, {res.url}')

    try:
        res = fetch_list('인공지능', page*10+1)
        news = parse_list(res)
        stats['ok'] += 1
    except Exception as e:
        logger.warning(f'에러 발생: {e}')
        stats['skip'] += 1
        continue
    finally:
        time.sleep(0.5) # 매너


    # urls.csv 에 저장
    with open('./urls.csv', 'a') as f:
        [f.write(n['링크']+','+'PEDNING\n') for n in news]

logger.info(
    f'현재까지 성공: {stats["ok"]}, 실패: {stats["failed"]}'\
    f' 스킵: {stats["skip"]}')

네이버 뉴스 목록 수집:   0%|          | 0/10 [00:00<?, ?it/s]2026-08-25 13:22:43,060 [WARNING] parsing 에러, 'NoneType' object is not subscriptable, https://s.search.naver.com/p/newssearch/3/api/tab/more?abt=null&de=2026.08.25&ds=2026.08.25&eid=&field=0&force_original=&is_dts=0&is_sug_officeid=0&mynews=0&news_office_checked=&nlu_query=&nqx_theme=%7B%22theme%22%3A%7B%22main%22%3A%7B%22name%22%3A%22encyclopedia%22%2C%22source%22%3A%22TOS%22%7D%7D%7D&nso=so%3Add%2Cp%3Aall%2Ca%3Aall&nx_and_query=&nx_search_hlquery=&nx_search_query=&nx_sub_query=&office_category=0&office_section_code=0&office_type=0&pd=0&photo=0&qdt=0&query=%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5&query_original=&rev=0&service_area=0&sm=tab_smr&sort=1&spq=0&ssc=tab.news.all&start=1
2026-08-25 13:22:43,062 [INFO] 현재까지 성공: 0, 실패: 0 스킵: 1
네이버 뉴스 목록 수집: 100%|██████████| 10/10 [00:06<00:00,  1.54it/s]
